# 02 — Machine Learning Workflow

**Sprint 7 — Machine Learning Fundamentals for AI/ML Engineers**  
**Notebook 2 of 20**

This notebook explains the complete Machine Learning workflow from
problem definition to post-deployment monitoring.

The concepts are explained using the **California Housing dataset**
and connected to the California Housing capstone project.

## Machine Learning Workflow

A real-world Machine Learning project follows a sequence of stages:

1. Problem Definition & Business Understanding
2. Data Collection
3. Data Understanding / EDA
4. Data Preprocessing
5. Feature Engineering & Feature Selection
6. Train/Test Split
7. Model Selection
8. Model Training
9. Prediction
10. Evaluation
11. Hyperparameter Tuning
12. Final Model Selection
13. Deployment
14. Monitoring

The important idea is that Machine Learning is **not just training a model**.

A model is only one part of the complete ML workflow.

## 1. Problem Definition & Business Understanding

Before working with data, we need to clearly define the problem.

We should answer:

- What business decision will the model support?
- What does success mean from a business perspective?
- Is Machine Learning the right solution?
- What is the cost of a wrong prediction?
- How will the prediction be used?

### California Housing Project

The goal is to:

> Estimate the median house value of a district using its
> socio-economic and geographic characteristics.

The target variable is:

`median_house_value`

This is a continuous numerical value.

Therefore, this is a **Regression problem**.

In [1]:
# Example: Identify the target variable

target = "median_house_value"

print("Target variable:", target)
print("Problem type: Regression")

Target variable: median_house_value
Problem type: Regression


## 2. Data Collection

Data collection means gathering the raw data required for the ML problem.

Possible sources include:

- Databases
- APIs
- Public datasets
- Web scraping
- Sensors
- Company systems
- External data providers

### California Housing Project

The California Housing dataset can be obtained using
`fetch_california_housing()` from Scikit-learn.

The dataset contains information about California districts,
including:

- Median income
- House age
- Number of rooms
- Number of bedrooms
- Population
- Households
- Latitude
- Longitude

The target is the median house value.

In [2]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)

housing.frame.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 3. Data Understanding / Exploratory Data Analysis

EDA means **Exploratory Data Analysis**.

Before building a model, we need to understand the data.

We check:

- Number of rows and columns
- Data types
- Missing values
- Statistical summaries
- Distributions
- Outliers
- Correlations
- Relationships between features
- Target distribution

### California Housing Project

Important observations include:

- Distribution of numerical features
- Distribution of the target
- Missing values
- Correlation between features and the target
- Geographic patterns using latitude and longitude

EDA helps us understand the dataset before preprocessing it.

In [3]:
# Dataset shape

print("Rows and columns:", housing.frame.shape)

Rows and columns: (20640, 9)


In [4]:
# Basic information

housing.frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


## 4. Data Preprocessing

Raw data usually cannot be directly given to a Machine Learning algorithm.

Preprocessing prepares the data for the model.

Common preprocessing steps include:

- Handling missing values
- Handling outliers
- Encoding categorical variables
- Feature scaling
- Data transformation

### California Housing Project

Examples:

- Missing values can be handled using `SimpleImputer`.
- Categorical variables such as `ocean_proximity` can be converted
  using One-Hot Encoding.
- Numerical features can be standardized using `StandardScaler`.

A `Pipeline` can combine these steps so that the same preprocessing
is consistently applied to training and test data.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

## 5. Feature Engineering & Feature Selection

### Feature Engineering

Feature engineering means creating new features from existing features
to provide more useful information to the model.

For example:

`rooms_per_household = total_rooms / households`

This may be more meaningful than using `total_rooms` alone.

### California Housing Project

Engineered features include:

- `rooms_per_household`
- `bedrooms_per_room`
- `population_per_household`

These ratios provide additional information about the characteristics
of each household or district.

### Feature Selection

Feature selection means identifying useful features and removing
features that are:

- Redundant
- Weakly related to the target
- Noisy
- Unnecessary

This can reduce complexity and improve model performance.

## 6. Train/Test Split

We divide the dataset into two parts:

### Training Set

Used to:

- Train the model
- Learn patterns
- Learn model parameters

### Test Set

Used later to:

- Evaluate the final model
- Estimate performance on unseen data

A common approach is:

**80% → Training data**

**20% → Test data**

The test set should not be used during model training or
hyperparameter selection.

In [7]:
from sklearn.model_selection import train_test_split

X = housing.frame.drop(columns=["MedHouseVal"])
y = housing.frame["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16512
Testing samples: 4128


## 7. Model Selection

After understanding and preparing the data, we choose suitable
Machine Learning algorithms.

For a regression problem, possible models include:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Support Vector Regression

### California Housing Project

Different regression models were compared:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Starting with multiple baseline models helps us understand
which type of algorithm works well for the problem.

## 8. Model Training

Training means allowing the algorithm to learn patterns from
the training data.

In Scikit-learn, training is usually performed using:

`model.fit(X_train, y_train)`

During training, the model learns parameters that help minimize
the selected loss function.

In [8]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ 0.45, 0.01,-0.12,...,-0. ,-0.42,-0.43]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-37.02
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(8)


## 9. Prediction

After training, the model can make predictions for unseen data.

The `predict()` method generates predicted values.

For example:

`model.predict(X_test)`

The model uses the patterns learned during training to estimate
the target values for the test data.

In [9]:
y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

First 10 predictions:
[0.71912284 1.76401657 2.70965883 2.83892593 2.60465725 2.01175367
 2.64550005 2.16875532 2.74074644 3.91561473]


## 10. Evaluation

Evaluation tells us how well the model performs.

For regression problems, common metrics include:

- MAE — Mean Absolute Error
- MSE — Mean Squared Error
- RMSE — Root Mean Squared Error
- R² — Coefficient of Determination

### RMSE

RMSE tells us approximately how far the predictions are from
the actual values, while giving larger errors more weight.

The important point is:

> The model should be evaluated on data that it did not train on.

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 0.5332001304956555
MSE : 0.5558915986952442
RMSE: 0.7455813830127763
R²  : 0.575787706032451


## 11. Hyperparameter Tuning

Machine Learning models have settings called **hyperparameters**.

These values are not learned directly from the training data.

Examples:

- Number of trees in Random Forest
- Maximum depth of a Decision Tree
- `C` in SVM
- `gamma` in RBF SVM

Instead of manually guessing these values, we can search through
different combinations.

Common techniques include:

- Grid Search
- Randomized Search

Cross-validation is commonly used during this process.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    "n_estimators": [50, 100],
    "max_features": [0.5, 1.0]
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

## 12. Final Model Selection

After hyperparameter tuning, we compare the candidate models.

The final model should consider:

- Predictive performance
- Model complexity
- Training time
- Prediction time
- Interpretability
- Business requirements

The test set should be used only for the final evaluation.

### Important

We should not repeatedly use the test set to decide which model
or hyperparameters are best.

Otherwise, information from the test set can influence our decisions,
which makes the final evaluation less trustworthy.

## 13. Deployment

A trained model becomes useful when it is available for real-world use.

Possible deployment methods include:

- REST API
- Batch prediction system
- Web application
- Internal application
- Cloud service

A Scikit-learn model can also be saved using tools such as
`joblib`.

Example workflow:

Training → Save Model → Deploy → Receive New Data → Predict

In [ ]:
import joblib

joblib.dump(model, "housing_model.pkl")

print("Model saved successfully.")

## 14. Monitoring

Machine Learning does not end after deployment.

The real world can change.

For example, a housing model trained using older housing data
may become less accurate when the housing market changes.

This can result from:

### Data Drift

The distribution of input data changes.

Example:

The income distribution of new districts is very different
from the training data.

### Concept Drift

The relationship between the input features and target changes.

Example:

The relationship between income and house prices changes
because the housing market changes.

Therefore, deployed models should be monitored continuously.

Monitoring may include:

- Input distributions
- Prediction distributions
- Actual prediction errors
- Data quality
- Model performance

If performance decreases significantly, the model may need
to be retrained.